# 🛡️ CyberShield AI - Network Intrusion Detection System
### Hybrid Deep Learning (CNN + LSTM) for Network Security

**Author**: Your Name  
**Course**: CSE496 - Ethical Hacking & Cybersecurity  
**Institution**: BRAC University

---

## 🚀 Quick Start
1. Click **Runtime** → **Run all**
2. Wait ~5 minutes for training
3. Download results from Files tab

---

## 📦 Install Dependencies

In [ ]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn tensorflow
print("✅ Dependencies installed!")

## 📥 Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("🛡️  CyberShield AI - Network Intrusion Detection System")
print("="*70)

## 📊 Load & Preprocess Data

In [ ]:
print("\n[1/6] Loading dataset...")

columns = ['duration', 'protocol_type', 'service', 'flag', 'src_bytes',
           'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
           'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
           'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
           'num_access_files', 'num_outbound_cmds', 'is_host_login',
           'is_guest_login', 'count', 'srv_count', 'serror_rate',
           'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate',
           'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate',
           'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate',
           'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
           'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
           'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
           'dst_host_srv_rerror_rate', 'label']

url = 'http://kdd.ics.uci.edu/databases/kddcup99/kddcup.data_10_percent.gz'
df = pd.read_csv(url, names=columns)

print(f"   ✓ Loaded {len(df)} records")

# Binary classification
df['binary_label'] = df['label'].apply(lambda x: 0 if x == 'normal.' else 1)
print(f"   ✓ Normal: {len(df[df['binary_label']==0])}")
print(f"   ✓ Attack: {len(df[df['binary_label']==1])}")

## 🔧 Feature Engineering

In [ ]:
print("\n[2/6] Feature engineering...")

categorical_cols = ['protocol_type', 'service', 'flag']
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df_encoded.drop(['label', 'binary_label'], axis=1).values
y = df_encoded['binary_label'].values

print(f"   ✓ Features: {X.shape[1]}")

## ⚙️ Prepare Training Data

In [ ]:
print("\n[3/6] Preparing data...")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Reshape for CNN-LSTM
X_train = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
X_test = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])

print(f"   ✓ Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

## 🏗️ Build Hybrid CNN-LSTM Model

In [ ]:
print("\n[4/6] Building model...")

model = models.Sequential([
    layers.Conv1D(64, 1, activation='relu', input_shape=(1, X_train.shape[2])),
    layers.MaxPooling1D(1),
    layers.Dropout(0.3),
    
    layers.LSTM(64, return_sequences=True),
    layers.Dropout(0.3),
    layers.LSTM(32),
    layers.Dropout(0.3),
    
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 🎯 Train Model

In [ ]:
print("\n[5/6] Training model...")

early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=128,
    callbacks=[early_stopping],
    verbose=1
)

print("   ✓ Training completed!")

## 📊 Evaluate Performance

In [ ]:
print("\n[6/6] Evaluating...")

y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("\n" + "="*70)
print("📊 RESULTS")
print("="*70)
print(f"\n🎯 Accuracy: {accuracy*100:.2f}%\n")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Attack']))

## 📈 Visualize Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Train', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val', linewidth=2)
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Normal', 'Attack'],
            yticklabels=['Normal', 'Attack'])
axes[1].set_title('Confusion Matrix', fontsize=14, fontweight='bold')
axes[1].set_ylabel('True')
axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.savefig('results.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Done! Download 'results.png' from Files tab")

## 💾 Save Model

In [ ]:
model.save('cybershield_model.h5')
print("\n💾 Model saved as 'cybershield_model.h5'")
print("\n" + "="*70)
print("✅ CyberShield AI - Completed Successfully!")
print("="*70)